In [1]:
from os import EX_DATAERR

# imports
import torch
import shutil
import pickle
import pandas as pd
import numpy as np
from mpmath.libmp import to_int
from torch import nn
from pathlib import Path
from collections import OrderedDict
from kagglehub import dataset_download
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from torch.nn import Sigmoid
from torchvision import transforms, datasets
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset, DataLoader, WeightedRandomSampler
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

/Users/tylerdow/PycharmProjects/Machine_Learning_OreoFinder/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# download dataset

local_data_path = Path("../data")
dataset_base_path = Path(dataset_download("crawford/cat-dataset"))
dataset_base_path = dataset_base_path / "cats"
oreo_path = local_data_path / "oreo"
not_oreo_path = local_data_path / "not_oreo"

oreo_path.mkdir(exist_ok=True)
not_oreo_path.mkdir(exist_ok=True)

for image in local_data_path.glob("*.jpg"):
    shutil.move(str(image), str(oreo_path / image.name))

for subdir in dataset_base_path.iterdir():
    if subdir.is_dir():
        for image in subdir.glob("*.jpg"):
            new_name = f"{subdir.name}_{image.name}"
            shutil.move(str(image), str(not_oreo_path / new_name))

In [3]:
# Split
full_dataset = datasets.ImageFolder(local_data_path)

x_data = []
y_data = []

for image, label in full_dataset:
    x_data.append(image)
    y_data.append(label)

# 80% Train
x_train, x_test, y_train, y_test = train_test_split(
    x_data,
    y_data,
    test_size=0.2,
    stratify=y_data,
    random_state=42
)

# 10% validation
# 10% test
x_train, x_val, y_train, y_val = train_test_split(
    x_train,
    y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=42
)

In [4]:
# preprocessing

# Resize only for KNN and Logistic Regression
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

x_train_proc = []
x_val_proc = []
x_test_proc = []

for photo in x_train:
    x_train_proc.append(transform(photo))

for photo in x_val:
    x_val_proc.append(transform(photo))

for photo in x_test:
    x_test_proc.append(transform(photo))

# Resize, Augment, and Normalize RGB
cnn_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2), # random brightness
    transforms.ToTensor()
])

cnn_x_train_proc = []

for photo in x_train:
    cnn_x_train_proc.append(cnn_transform(photo))


KeyboardInterrupt: 

In [ ]:
# Weighted Random Sampler
#train_targets = targets[train_idx]
#class_sample_count = np.array([len(np.where(train_targets == t)[0]) for t in np.unique(train_targets)])
#weight = 1. / class_sample_count
#samples_weight = np.array([weight[t] for t in train_targets])
#samples_weight = torch.from_numpy(samples_weight)

#sampler = WeightedRandomSampler(
#    weights=samples_weight,
#    num_samples=len(samples_weight),
#    replacement=True
#)

#train_loader = DataLoader(train_dataset, batch_size=32, sampler=sampler, num_workers=4)
#cnn_train_loader = DataLoader(cnn_x_train_proc, batch_size=32, num_workers=4)
#val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)
#test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4)

#images, labels = next(iter(train_loader))

In [ ]:
# flatten images, convert to numpy

# train
x_train_flat = []

for image in x_train_proc:
    x_train_flat.append(image.view(-1).numpy())

x_train = np.vstack(x_train_flat)

# validation
x_val_flat = []

for image in x_val_proc:
    x_val_flat.append(image.view(-1).numpy())

x_val_flat = np.vstack(x_val_flat)

# test
x_test_flat = []

for image in x_test_proc:
    x_test_flat.append(image.view(-1).numpy())

x_test_flat = np.vstack(x_test_flat)

In [ ]:
#PCA on the images
pca = PCA(n_components=0.95)
x_train = pca.fit_transform(x_train)
x_val_new = pca.transform(x_val_flat)
x_test_new = pca.transform(x_test_flat)

In [ ]:
# hyperparameter tuning
C_values = [0.01, 0.1, 1]
log_reg_model = None
best_f1 = 0
best_C = None

# f1 validation
for C in C_values:
    regression = LogisticRegression(max_iter=1, solver='saga', C=C, class_weight='balanced')

    regression.fit(x_train, y_train)
    val_preds = regression.predict(x_val_new)

    f1 = f1_score(y_val, val_preds, average='macro')
    accuracy = accuracy_score(y_val, val_preds)

    print(f"C={C}, Validation F1={f1}, Validation Accuracy: {accuracy}")
    if f1 > best_f1:
        best_f1 = f1
        log_reg_model = regression
        best_C = C

print(f"\nBest C: {best_C} with F1={best_f1}")

predictions = log_reg_model.predict(x_test_new)
print(f"Predictions: {predictions}")
print(f"Actual labels: {y_test}")
print(f"Accuracy: {accuracy_score(y_test, predictions)}")

In [ ]:
# K-NN

# Validation
k_val = [1, 2, 3, 5, 7, 9]
best_k = None
best_f1 = 0
for k in k_val:
    model = KNeighborsClassifier(n_neighbors=k, n_jobs=-1)
    model.fit(x_train, y_train)
    val_predictions = model.predict(x_val_new)
    f1 = f1_score(y_val, val_predictions, average='macro')
    print(f"k={k}, Validation F1 Score: {f1:.4f}")
    if f1 > best_f1:
        best_f1 = f1
        best_k = k

# Training with best k
print(f"Best k: {best_k} with F1 Score: {best_f1:.4f}")
knn_model = KNeighborsClassifier(n_neighbors=best_k, n_jobs=-1)
knn_model.fit(x_train, y_train)

# Printing predictions and actual labels
predictions = knn_model.predict(x_test_new)
print(f"Predictions: {predictions}")
print(f"Actual labels: {y_test}")

accuracy = np.mean(predictions == y_test)
print(f"Test Accuracy: {accuracy:.4f}")

# Showing incorrect predictions and their neighbors
incorrect = np.where(((predictions == 1) & (y_test == 0)) | ((predictions == 0) & (y_test == 1)))[0] 
num_mistakes = len(incorrect) 
if num_mistakes > 0: 
    fig, axes = plt.subplots(num_mistakes, best_k + 1, figsize=(20, 4 * num_mistakes)) 
   
    if num_mistakes == 1: 
        axes = np.expand_dims(axes, axis=0) 
    for i, test_idx in enumerate(incorrect): 
        sample_image = x_test_new[test_idx]
        distances,  neighbor_indices = knn_model.kneighbors(sample_image.reshape(1, -1), n_neighbors=best_k)
        ax_test = axes[i, 0]
        img_reshaped = sample_image.reshape(3, 128, 128).transpose(1, 2, 0) 
        ax_test.imshow(img_reshaped) 
        ax_test.set_title(f"INCORRECT\n(Index {test_idx})", color='red')  
        ax_test.axis('off')
        for j, train_idx in enumerate(neighbor_indices[0]): 
            ax_nb = axes[i, j + 1] 
            neighbor_img = x_train[train_idx].reshape(3, 128, 128).transpose(1, 2, 0)
            ax_nb.imshow(neighbor_img) 
            ax_nb.set_title(f"Neighbor {j+1}\nDist: {distances[0][j]:.2f}") 
            ax_nb.axis('off')

plt.tight_layout() 
plt.show()


In [ ]:
# CNN
learning_rate = 0.001
epochs = 1

cnn_model = nn.Sequential(
    OrderedDict([
        ("conv1", nn.Conv2d(3, 128, kernel_size=3, padding=1)),
        ("relu1", nn.ReLU()),
        ("pool1", nn.MaxPool2d(kernel_size=2, stride=2)),
        ("conv2", nn.Conv2d(128, 64, kernel_size=3, padding=1)),
        ("relu2", nn.ReLU()),
        ("pool2", nn.MaxPool2d(kernel_size=2, stride=2)),
        ("conv3", nn.Conv2d(64, 32, kernel_size=3, padding=1)),
        ("relu3", nn.ReLU()),
        ("pool3", nn.MaxPool2d(kernel_size=2, stride=2)),
        ("flatten", nn.Flatten()),
        ("linear1", nn.Linear(32 * 16 * 16, 128)),
        ("relu5", nn.ReLU()),
        ("linear2", nn.Linear(in_features=128, out_features=1)),
    ])
).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(cnn_model.parameters(), lr=learning_rate)

cnn_model.train()
for epoch in range(epochs):
    total_loss = 0

    outputs = []

    for i, image in enumerate(cnn_x_train_proc):
        images = image.unsqueeze(0).to(device)

        optimizer.zero_grad()

        outputs.append(cnn_model(images).squeeze(1))

    loss = criterion(outputs, y_train)
    total_loss += loss.item()

    loss.backward()
    optimizer.step()

    print(f"epoch {epoch + 1}/{epochs}")
    print(f"epoch {total_loss:.2f}")

In [ ]:
# Save the models
local_path = Path("../models")

log_reg_path = Path(local_path / "logRegModel.pkl")
with open(log_reg_path, 'wb') as f:
    pickle.dump(log_reg_model, f)

knn_path = Path(local_path / "knnModel.pkl")
with open(knn_path, 'wb') as f:
    pickle.dump(knn_model, f)

cnn_path = Path(local_path / "cnnModel.pkl")
with open(cnn_path, 'wb') as f:
    pickle.dump(cnn_model, f)

In [ ]:
# Load the models
local_path = Path("../models")

log_reg_path = Path(local_path / "logRegModel.pkl")
with open(log_reg_path, 'rb') as f:
    log_reg_model = pickle.load(f)

knn_path = Path(local_path / "knnModel.pkl")
with open(knn_path, 'rb') as f:
    knn_model = pickle.load(f)

cnn_path = Path(local_path / "cnnModel.pkl")
with open(cnn_path, 'rb') as f:
    cnn_model = pickle.load(f)

In [ ]:
# evaluation

models = pd.DataFrame(index=["Accuracy", "Precision", "Recall", "F1", "AUC"])

lr_accuracy = accuracy_score(y_test, log_reg_model.predict(x_test_new))
lr_precision = precision_score(y_test, log_reg_model.predict(x_test_new))
lr_recall = recall_score(y_test, log_reg_model.predict(x_test_new))
lr_f1 = f1_score(y_test, log_reg_model.predict(x_test_new))
lr_auc = roc_auc_score(y_test, log_reg_model.predict(x_test_new))
lr_roc = roc_curve(y_test, log_reg_model.predict(x_test_new))

models["Logistic Regression"] = [lr_accuracy, lr_precision, lr_recall, lr_f1, lr_auc]

knn_accuracy = accuracy_score(y_test, knn_model.predict(x_test_new))
knn_precision = precision_score(y_test, knn_model.predict(x_test_new))
knn_recall = recall_score(y_test, knn_model.predict(x_test_new))
knn_f1 = f1_score(y_test, knn_model.predict(x_test_new))
knn_auc = roc_auc_score(y_test, knn_model.predict(x_test_new))
knn_roc = roc_curve(y_test, knn_model.predict(x_test_new))

models["Logistic Regression"] = [knn_accuracy, knn_precision, knn_recall, knn_f1, knn_auc]

predictions = []

for image, label in x_test:
    probability = cnn_model(image)
    prediction = to_int(Sigmoid(probability > 0.5))

    predictions.append(prediction)

cnn_accuracy = accuracy_score(y_test, predictions)
cnn_precision = precision_score(y_test, predictions)
cnn_recall = recall_score(y_test, predictions)
cnn_f1 = f1_score(y_test, predictions)
cnn_auc = roc_auc_score(y_test, predictions)
cnn_roc = roc_curve(y_test, predictions)

models["Logistic Regression"] = [cnn_accuracy, cnn_precision, cnn_recall, cnn_f1, cnn_auc]